# Margin distributions (A1)

Where the score-separation margins actually are, and how much of the mass is at
exactly zero.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
from tfidf_stability.analysis.summarise import summarise_values
from tfidf_stability.datasets.loaders import load_dataset
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.ranking.margins import boundary_margin
from tfidf_stability.similarity.cosine import cosine_against_corpus
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

data = load_dataset("synthetic_tiny")
pipeline = PreprocessingPipeline()
features = [pipeline.preprocess(str(r["text"])) for r in data.records]
model = TfidfVectoriser().fit(features, data.doc_ids)
documents = [model.document(i) for i in range(model.n_documents)]

vectors = [
    sorted(cosine_against_corpus(
        TfidfVectoriser.transform_query(list(f)[:6], model), documents, model.norms),
        reverse=True)
    for f in features[::4]
]
print(f"{len(vectors)} queries over {model.n_documents} documents")

In [ ]:
for k in (1, 5, 10, 20, 50):
    if k >= model.n_documents:
        continue
    margins = [boundary_margin(v, k) for v in vectors]
    d = summarise_values(f"m_{k}", [m.value for m in margins if m.defined])
    print(f"k={k:3}  n={d.n:4}  exact ties {d.share_zero:6.1%}  "
          f"p50={d.percentiles['p50']:.3e}  p95={d.percentiles['p95']:.3e}")

Percentiles are **nearest-rank**: every reported value is an observation that
actually occurred. Interpolating would invent a margin no query produced, which
could not be looked up in the raw data.

The exact-tie share is reported separately because it vanishes into the
percentiles once it exceeds 50%.

## The gap distribution, and the empty interval

Adjacent gaps are either exactly zero or well above 1e-9. The interval between is
empty -- so at fine tau the "near-tie" regime is really the **exact**-tie regime.
See `docs/spec_addenda.md#g22`.

In [ ]:
from itertools import pairwise

gaps = [a - b for v in vectors for a, b in pairwise(v)]
buckets = [
    ("exactly 0", sum(g == 0.0 for g in gaps)),
    ("(0, 1e-9)", sum(0.0 < g < 1e-9 for g in gaps)),
    ("[1e-9, 1e-6)", sum(1e-9 <= g < 1e-6 for g in gaps)),
    ("[1e-6, 1e-3)", sum(1e-6 <= g < 1e-3 for g in gaps)),
    (">= 1e-3", sum(g >= 1e-3 for g in gaps)),
]
for name, count in buckets:
    print(f"  {name:14} {count:6}  ({count / len(gaps):6.1%})")

positive = [g for g in gaps if g > 0]
print(f"\nsmallest strictly-positive gap: {min(positive):.3e}")